# Customer Churn Analysis - Team 1
## Data Preparation & Preprocessing (Data Engineer role)

This notebook:
1. Loads the raw dataset
2. Checks data integrity (missing values, duplicates, logical consistency)
3. Encodes categorical variables
4. Splits into train/test sets
5. Scales numeric features
6. Saves all output files

**Run all cells top to bottom.** Make sure `Dataset_ATS_v2.csv` is in the same folder as this notebook.

## 0. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2

## 1. Load the raw data

In [2]:
df = pd.read_csv("Dataset_ATS_v2.csv")
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (7043, 10)


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   gender           7043 non-null   str  
 1   SeniorCitizen    7043 non-null   int64
 2   Dependents       7043 non-null   str  
 3   tenure           7043 non-null   int64
 4   PhoneService     7043 non-null   str  
 5   MultipleLines    7043 non-null   str  
 6   InternetService  7043 non-null   str  
 7   Contract         7043 non-null   str  
 8   MonthlyCharges   7043 non-null   int64
 9   Churn            7043 non-null   str  
dtypes: int64(3), str(7)
memory usage: 550.4 KB


## 2. Data Integrity & Consistency Checks

### 2a. Missing values

In [4]:
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64

Total missing values: 0


In [5]:
# Defensive imputation, in case a future data pull has gaps:
#   numeric -> median, categorical -> mode
for col in df.select_dtypes(include=["int64", "float64"]).columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed {col} with median {median_val}")

for col in df.select_dtypes(include=["object", "string"]).columns:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Imputed {col} with mode '{mode_val}'")

### 2b. Duplicate rows

In [6]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Removed duplicates. New shape: {df.shape}")

Duplicate rows: 302
Removed duplicates. New shape: (6741, 10)


### 2c. Logical consistency check

`MultipleLines = Yes` should never occur when `PhoneService = No` -- you can't have
multiple phone lines without phone service in the first place.

In [7]:
inconsistent_mask = (df["PhoneService"] == "No") & (df["MultipleLines"] == "Yes")
print(f"Inconsistent rows found: {inconsistent_mask.sum()}")

df.loc[inconsistent_mask, "MultipleLines"] = "No"
print("Corrected.")

Inconsistent rows found: 256
Corrected.


### 2d. Range checks on numeric columns

In [8]:
range_checks = {
    "tenure": (0, 100),
    "MonthlyCharges": (0, 500),
    "SeniorCitizen": (0, 1),
}
for col, (low, high) in range_checks.items():
    out_of_range = df[(df[col] < low) | (df[col] > high)]
    print(f"{col}: {len(out_of_range)} values outside expected range [{low}, {high}]")

tenure: 0 values outside expected range [0, 100]
MonthlyCharges: 0 values outside expected range [0, 500]
SeniorCitizen: 0 values outside expected range [0, 1]


## 3. Encode categorical variables

- Binary columns -> label-encoded to 0/1
- `Contract` (3 nominal categories, no numeric order) -> one-hot encoded

In [9]:
binary_map = {"Yes": 1, "No": 0}
binary_cols = ["Dependents", "PhoneService", "MultipleLines", "Churn"]
for col in binary_cols:
    df[col] = df[col].map(binary_map)

df["gender"] = df["gender"].map({"Male": 1, "Female": 0})
df["InternetService"] = df["InternetService"].map({"DSL": 0, "Fiber optic": 1})

df = pd.get_dummies(df, columns=["Contract"], prefix="Contract", drop_first=False)
contract_cols = [c for c in df.columns if c.startswith("Contract_")]
for c in contract_cols:
    df[c] = df[c].astype(int)

print("Encoded columns:", df.columns.tolist())
df.head()

Encoded columns: ['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'MonthlyCharges', 'Churn', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year']


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,MonthlyCharges,Churn,Contract_Month-to-month,Contract_One year,Contract_Two year
0,0,0,0,1,0,0,0,25,1,1,0,0
1,1,0,0,41,1,0,0,25,0,0,1,0
2,0,0,1,52,1,0,0,19,0,1,0,0
3,0,0,0,1,1,0,0,76,1,0,1,0
4,1,0,0,67,1,0,1,51,0,1,0,0


## 4. Save the full cleaned + encoded dataset

This is the unscaled version -- useful for teammates doing EDA or model types
(e.g. tree-based models) that don't need scaled features.

In [10]:
df.to_csv("Dataset_ATS_v2_preprocessed.csv", index=False)
print("Saved: Dataset_ATS_v2_preprocessed.csv")

Saved: Dataset_ATS_v2_preprocessed.csv


## 5. Train / test split

Stratified on `Churn` so both sets keep the same churn rate as the full dataset.

In [11]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate - train: {y_train.mean():.2%}, test: {y_test.mean():.2%}, full: {y.mean():.2%}")

Train: (5392, 11), Test: (1349, 11)
Churn rate - train: 26.58%, test: 26.54%, full: 26.57%


## 6. Feature scaling

`StandardScaler` is fit on the **training set only**, then applied to the test set.
This avoids data leakage -- if you fit the scaler on the full dataset first, information
from the test set would "leak" into training.

In [12]:
numeric_cols = ["tenure", "MonthlyCharges"]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train_scaled.head()

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,MonthlyCharges,Contract_Month-to-month,Contract_One year,Contract_Two year
4231,0,0,1,0.944773,1,1,1,1.486201,1,0,0
943,0,0,0,1.232687,1,0,0,-0.234838,1,0,0
5353,0,0,0,0.821382,1,0,0,0.946267,0,1,0
287,1,0,1,1.602861,1,0,0,1.621184,1,0,0
3222,1,0,0,-0.988360,1,0,0,0.338842,1,0,0


## 7. Save train and test sets

In [13]:
train_out = X_train_scaled.copy()
train_out["Churn"] = y_train.values
train_out.to_csv("train_set.csv", index=False)

test_out = X_test_scaled.copy()
test_out["Churn"] = y_test.values
test_out.to_csv("test_set.csv", index=False)

print("Saved: train_set.csv", train_out.shape)
print("Saved: test_set.csv", test_out.shape)

Saved: train_set.csv (5392, 12)
Saved: test_set.csv (1349, 12)


## Summary

| File | Contents |
|---|---|
| `Dataset_ATS_v2_preprocessed.csv` | Full cleaned + encoded dataset, unscaled |
| `train_set.csv` | 80% of data, scaled, includes `Churn` target |
| `test_set.csv` | 20% of data, scaled, includes `Churn` target |

Next step (for whoever is doing feature engineering / modeling): load `train_set.csv`
and `test_set.csv` directly -- no further cleaning needed.